# Data and evidence

This notebook prepares page chunks and their source evidence links, then reads the
accepted question and reference records used in evaluation.

The default input is [RFC 2795](../examples/data/rfc2795.pdf), the 20-page Infinite
Monkey Protocol Suite published on 1 April 2000. Four illustrative questions cover
multi-page evidence, a protocol lookup, an explicit response rule and missing information.
Source and redistribution details are in [the data notes](../examples/data/README.md).

For an authorised PDF corpus, select a private source specification and output
folder in the first input cell. The [input contract](../docs/classical_rag_artifacts.md#prepare-a-pdf-corpus)
shows the document list and accepted question/reference fields. The same page
preparation and source links serve both system paths. Run private copies and
store their outputs outside Git, under `local/`.

Run linearly after the [README setup](../README.md). No model call is needed.


In [1]:
import hashlib
import json
from pathlib import Path

import pdfplumber
from IPython.display import Markdown, display
from tabulate import tabulate
from rag_comparison import config

## 1. Bind documents to a corpus snapshot

A **corpus snapshot** identifies a fixed source collection. A **document ID**
identifies one document within it. The PDF hash binds that document to exact bytes.
Use physical PDF page numbers, starting at 1, for source identity; printed labels
inside a document do not define the page identity.

The default source specification describes the public RFC. A private specification
contains `documents`, each with a unique `document_id`, a relative `pdf_file` and
its `sha256`. Paths are relative to the specification's folder. The recorded list
order defines document order; pages remain in their physical order.


In [2]:
source_file = Path("../examples/data/reference_packages.json")
input_dir = Path("../local/classical_rag/rfc2795-inputs")
# For a corpus, select e.g. ../local/corpus/reference_packages.json and a new output folder.

public_example = (
    source_file.resolve() == Path("../examples/data/reference_packages.json").resolve()
)
assert input_dir.resolve().is_relative_to(Path("../local").resolve())
if not public_example:
    assert source_file.resolve().is_relative_to(Path("../local").resolve())

source = json.loads(source_file.read_text(encoding="utf-8"))
snapshot_id = source["corpus_snapshot_id"]
documents = [source["document"]] if public_example else source["documents"]
assert documents and len({d["document_id"] for d in documents}) == len(documents)

for document in documents:
    pdf_path = source_file.parent / document["pdf_file"]
    assert hashlib.sha256(pdf_path.read_bytes()).hexdigest() == document["sha256"]
print({"snapshot": snapshot_id, "documents": len(documents), "PDF_hashes": "verified"})

{'snapshot': 'DEMO-RFC2795-001', 'documents': 1, 'PDF_hashes': 'verified'}


## 2. Extract one physical page at a time

Use `pdfplumber`'s default `extract_text()` on each PDF page. This baseline does
not apply OCR, remove headers or footers, dehyphenate words, reconstruct tables,
or join text across pages. Page qualification records text-layer and extraction
status and any blocker before chunking.

All 20 RFC pages have extractable text. An image-only page remains explicitly
blocked for OCR; it is not silently replaced by another extraction method. Empty
or unsupported pages remain in the raw page ledger and produce no chunk.
A library extraction error stays visible.


In [3]:
page_records = []
source_documents = []
for document in documents:
    pdf_path = source_file.parent / document["pdf_file"]
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)
        if "page_count" in document:
            assert page_count == document["page_count"]
        source_documents.append(
            {
                "document_id": document["document_id"],
                "sha256": document["sha256"],
                "page_count": page_count,
            }
        )
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            has_text_layer = bool(page.chars)
            page_records.append(
                {
                    "corpus_snapshot_id": snapshot_id,
                    "document_id": document["document_id"],
                    "page_number": page_number,
                    "element_type": "page",
                    "text_layer_status": "present" if has_text_layer else "absent",
                    "extraction_status": "extracted",
                    "blocker": (
                        "blocked_ocr_required"
                        if not has_text_layer and page.images
                        else None
                    ),
                    "text": text,
                }
            )
assert len(page_records) == sum(d["page_count"] for d in source_documents)
print({"documents": len(source_documents), "physical_pages": len(page_records)})
if public_example:
    print(page_records[0]["text"][:280])

{'documents': 1, 'physical_pages': 20}
Network Working Group S. Christey
Request for Comments: 2795 MonkeySeeDoo, Inc.
Category: Informational 1 April 2000
The Infinite Monkey Protocol Suite (IMPS)
Status of this Memo
This memo provides information for the Internet community. It does
not specify an Internet standard o


## 3. Normalise whitespace and qualify page chunks

The normalisation is exactly `" ".join(text.split())`: runs of spaces, tabs and
line breaks become one space. Page boundaries, headers, footers and ASCII
table/diagram characters are retained; their layout is not reconstructed.

A page produces a chunk only when its recorded status is supported and its
normalised text is nonempty. An OCR-required blocker remains blocked. This small
status function makes the eligibility rule visible.

In [4]:
def chunk_status(record: dict) -> str:
    if record["blocker"] == "blocked_ocr_required":
        return "blocked_ocr_required"
    if (
        record["element_type"] != "page"
        or record["text_layer_status"] != "present"
        or record["extraction_status"] != "extracted"
    ):
        return "blocked_unsupported_status"
    if not " ".join(record["text"].split()):
        return "skipped_empty_text"
    return "chunk"


for page in page_records:
    page["normalised_text"] = " ".join(page["text"].split())
    page["chunk_status"] = chunk_status(page)

if public_example:
    print(page_records[0]["normalised_text"][:280])

Network Working Group S. Christey Request for Comments: 2795 MonkeySeeDoo, Inc. Category: Informational 1 April 2000 The Infinite Monkey Protocol Suite (IMPS) Status of this Memo This memo provides information for the Internet community. It does not specify an Internet standard o


## 4. Keep chunks and evidence identity separate

The baseline uses **one chunk per eligible page**, with no overlap or cross-page
chunks. A chunk is a system input artifact. An **Evidence Unit** is the common
source identity used by both systems during evaluation:

```text
snapshot + document + physical page
               ↓
        page chunk → origin → Evidence Unit
```

The origin record binds the chunk to its source. Its Evidence Unit ID has the
form `eu::<snapshot>::<document>::p<physical-page>`.
A different snapshot or document is a different source identity, even at the same
page number. The corpus chunk ID includes snapshot, document and page. The fixed RFC example
retains its published artifact IDs for the saved public runs.

In [5]:
chunks = []
for page in page_records:
    if page["chunk_status"] == "chunk":
        chunks.append(
            {
                "chunk_id": (
                    f"DEMO-CHUNK-{page['page_number']:03d}"
                    if public_example
                    else f"chunk::{snapshot_id}::{page['document_id']}::p{page['page_number']}"
                ),
                "corpus_snapshot_id": page["corpus_snapshot_id"],
                "document_id": page["document_id"],
                "page_number": page["page_number"],
                "chunk_index_on_page": 1,
                "chunking_config_ref": "page-whitespace-v1",
                "text": page["normalised_text"],
            }
        )

origins = [
    {
        "chunk_id": chunk["chunk_id"],
        "corpus_snapshot_id": chunk["corpus_snapshot_id"],
        "document_id": chunk["document_id"],
        "page_number": chunk["page_number"],
        "projection_status": "projected",
        "origin_evidence_unit_ids": [
            f"eu::{chunk['corpus_snapshot_id']}::{chunk['document_id']}::p{chunk['page_number']}"
        ],
    }
    for chunk in chunks
]

## 5. Inspect and check the source links

Each chunk has exactly one origin, and its text equals its own normalised source
page. The baseline inputs of both systems preserve these texts. Classical RAG
retrieves chunks; GraphRAG links its text units back through the input and origin
records. Their system-specific artifact IDs do not replace the shared page identity.
Segmentation changes can create several chunks from one page while retaining the
same source Evidence Unit.

In [6]:
pages_by_source = {
    (page["document_id"], page["page_number"]): page for page in page_records
}
assert len(pages_by_source) == len(page_records)
assert chunks and len(chunks) == len(origins)
assert len(chunks) == sum(page["chunk_status"] == "chunk" for page in page_records)
assert len({chunk["chunk_id"] for chunk in chunks}) == len(chunks)
for chunk, origin in zip(chunks, origins, strict=True):
    page = pages_by_source[chunk["document_id"], chunk["page_number"]]
    assert chunk["chunk_id"] == origin["chunk_id"]
    assert chunk["chunk_index_on_page"] == 1
    assert chunk["text"] == page["normalised_text"]
    assert origin["corpus_snapshot_id"] == chunk["corpus_snapshot_id"]
    assert origin["document_id"] == chunk["document_id"]
    assert origin["page_number"] == chunk["page_number"]
    assert len(origin["origin_evidence_unit_ids"]) == 1

rows = [
    (
        c["document_id"],
        c["page_number"],
        c["chunk_id"],
        o["origin_evidence_unit_ids"][0],
    )
    for c, o in zip(chunks, origins, strict=True)
]
display(
    Markdown(
        tabulate(
            rows[:5],
            headers=["Document", "Physical page", "Chunk", "Evidence Unit"],
            tablefmt="github",
        )
    )
)
print(
    {
        "physical_pages": len(page_records),
        "page_chunks": len(chunks),
        "excluded_pages": len(page_records) - len(chunks),
    }
)

| Document   |   Physical page | Chunk          | Evidence Unit                      |
|------------|-----------------|----------------|------------------------------------|
| RFC-2795   |               1 | DEMO-CHUNK-001 | eu::DEMO-RFC2795-001::RFC-2795::p1 |
| RFC-2795   |               2 | DEMO-CHUNK-002 | eu::DEMO-RFC2795-001::RFC-2795::p2 |
| RFC-2795   |               3 | DEMO-CHUNK-003 | eu::DEMO-RFC2795-001::RFC-2795::p3 |
| RFC-2795   |               4 | DEMO-CHUNK-004 | eu::DEMO-RFC2795-001::RFC-2795::p4 |
| RFC-2795   |               5 | DEMO-CHUNK-005 | eu::DEMO-RFC2795-001::RFC-2795::p5 |

{'physical_pages': 20, 'page_chunks': 20, 'excluded_pages': 0}


## 6. Distinguish answerability from evaluation role

**Answerability** states whether the fixed corpus supports the requested answer.
The **benchmark role** determines where a question is evaluated.

| Group | Answerability | Role and meaning |
|---|---|---|
| A | `answerable` | `primary_answerable`: active retrieval and answer evaluation |
| U | `abstention_expected` | `robustness_abstention`: active test of appropriate abstention |
| Deferred | Kept as recorded | `deferred`: held outside active evaluation until its reference is reliable |

Deferred is a role, not a third answerability value. U means that the required
information is absent from the fixed source corpus, rather than unknowable in general.
Question/reference preparation uses Azure-assisted drafting and source-based review;
accepted references define evaluation. The example records below let you inspect
that contract without a model call.

In [7]:
questions = source["questions"]
assert questions and len({q["question_id"] for q in questions}) == len(questions)
answerable = next(
    (q for q in questions if q["benchmark_role"] == "primary_answerable"), None
)
unanswerable = next(
    (q for q in questions if q["benchmark_role"] == "robustness_abstention"), None
)
rows = [(q["question_id"], q["answerability"], q["benchmark_role"]) for q in questions]
display(
    Markdown(
        tabulate(rows, headers=["Question", "Answerability", "Role"], tablefmt="github")
    )
)
if public_example:
    print(answerable["question_text"])
    print(unanswerable["question_text"])

| Question        | Answerability       | Role                  |
|-----------------|---------------------|-----------------------|
| DEMO-IMPS-Q-001 | answerable          | primary_answerable    |
| DEMO-IMPS-Q-002 | answerable          | primary_answerable    |
| DEMO-IMPS-Q-003 | answerable          | primary_answerable    |
| DEMO-IMPS-Q-004 | abstention_expected | robustness_abstention |

Which numeric KEEPER request code stops all monkey business, which response codes may a SIMIAN use, and how can an attacker misuse that request?
What response timeout, in milliseconds, does CHIMP require for a SEND FOOD request?


## 7. Read the reference package

A reference answer is a readable summary. Correctness is checked against the
requirements and source evidence, not by exact string matching.

| Field | Purpose |
|---|---|
| `canonical_answer_summary` | Readable expected answer |
| `required_facts` | Individual facts needed for a complete answer |
| `acceptable_variants` | Equivalent wording that should be accepted |
| `prohibited_facts` | Incorrect or unsupported content boundaries |
| `numeric_tolerance` | A justified reference value, unit and allowed deviation, when applicable |
| `minimal_evidence_sets` | Reviewed combinations of source units sufficient for the answer |
| `qrels` | Accepted relevance grades for question/Evidence-Unit pairs |
| `abstention_expected`, `abstention_reason` | The expected response boundary |

The public example has no numeric tolerance; its empty field does not invent a
zero tolerance. Read any private question's recorded tolerance in its own units. Numeric, multi-evidence and exception-sensitive diagnostic properties
can overlap. They are stored in `diagnostic_tags` and `review_triggers`, rather
than defining separate, disjoint question populations.

In [8]:
reference = answerable["reference_package"] if answerable is not None else None
if public_example:
    print(reference["canonical_answer_summary"])
    for number, fact in enumerate(reference["required_facts"], start=1):
        print(f"Required fact {number}: {fact}")
    print("Acceptable wording:", reference["acceptable_variants"])
    print("Prohibited content:", reference["prohibited_facts"])
    print("Diagnostic tags:", answerable["diagnostic_tags"])

STOP is KEEPER request code 7 and stops all monkey business. A SIMIAN may answer with NORESPONSE, ALIVE, DEAD, or GONE. An attacker can send one STOP request to each SIMIAN to cause a massive denial of service throughout the ZOO.
Required fact 1: STOP is KEEPER request code 7 and stops all monkey business.
Required fact 2: A SIMIAN may answer STOP with NORESPONSE, ALIVE, DEAD, or GONE.
Required fact 3: An attacker can send one STOP request to each SIMIAN to cause a massive denial of service throughout the ZOO.
Acceptable wording: ['Return codes 4, 5, 6 and 2 correspond to NORESPONSE, ALIVE, DEAD and GONE.', 'Sending a STOP request to every SIMIAN can disrupt the entire ZOO.']
Prohibited content: ['A different numeric request code for STOP.', 'ACCEPT or REFUSE as a permitted STOP response.', 'The described STOP attack requires transcript transfer.']
Diagnostic tags: ['multi_evidence_and_semantics_assumed']


## 8. Separate relevance from sufficient evidence

The relevance scale uses **linear grades 0/1/2**:

| Grade | Meaning |
|---:|---|
| 0 | No contribution to answering the question |
| 1 | Related to the topic, without direct support for a Required Fact |
| 2 | Direct support for at least one Required Fact |

For the first public example question, page 7 identifies STOP and its numeric code, page 8 gives
the permitted responses, and page 16 describes its misuse. All three receive
grade 2; the minimal evidence set is `{page 7, page 8, page 16}`.
It is reviewed for sufficiency; it is not inferred simply from the grade-2 units.
Alternative complete sets can be stored when more than one combination suffices.

Absent judgements and failed projections are not artificial grade-0 judgements.
Both systems use the same question-wise relevance pool within a comparison;
PRIMARY and FINAL comparisons have their respective pool versions.
For a corpus, validate the recorded pool and minimal sets against the extracted
source identities. The judged pool can be a subset of the corpus; unjudged pages
are not automatically assigned grade 0.


In [9]:
evidence_ids = {origin["origin_evidence_unit_ids"][0] for origin in origins}
assert len(evidence_ids) == len(origins)
for question in questions:
    reviewed = question["reference_package"]
    assert set(reviewed["qrels"]) <= evidence_ids
    assert all(grade in (0, 1, 2) for grade in reviewed["qrels"].values())
    assert all(
        set(units) <= evidence_ids for units in reviewed["minimal_evidence_sets"]
    )

if public_example:
    qrels = reference["qrels"]
    minimal_sets = [set(units) for units in reference["minimal_evidence_sets"]]
    assert set(qrels) == evidence_ids
    required_pages = [7, 8, 16]
    required_ids = {
        origins[p - 1]["origin_evidence_unit_ids"][0] for p in required_pages
    }
    assert minimal_sets == [required_ids]
    for unit in required_ids:
        assert qrels[unit] == 2
        assert not any(required <= {unit} for required in minimal_sets)
    print(
        {
            "required_source_pages": required_pages,
            "one_page_sufficient": False,
            "all_three_pages_sufficient": True,
            "judged_source_pages": len(qrels),
        }
    )
else:
    print(
        {
            "questions": len(questions),
            "source_units": len(evidence_ids),
            "reference_source_links": "verified",
        }
    )

{'required_source_pages': [7, 8, 16], 'one_page_sufficient': False, 'all_three_pages_sufficient': True, 'judged_source_pages': 20}


## 9. Keep the response boundary visible

The fourth public example question asks for a CHIMP SEND FOOD timeout in milliseconds. RFC 2795
defines the request and its responses, but gives no numeric timeout. This reference
therefore expects abstention. The third public example question remains answerable: the absence
of a PAN encryption-discovery procedure is explicitly stated in the source. U cases have no
minimal evidence sets for the primary retrieval measure. Required Facts can still
describe an answer boundary in another U reference; an empty list here is specific
to this example.

The separate **Genuine User** collection keeps users' original wording and its
own reviewed evidence boundaries. Its nominal answer categories form a separate
assessment; primary qrels, minimal evidence sets and ordinal answer grades are
not assigned to that collection by this example.

In [10]:
for question in questions:
    if question["benchmark_role"] == "robustness_abstention":
        u_reference = question["reference_package"]
        assert question["answerability"] == "abstention_expected"
        assert u_reference["abstention_expected"] is True
        assert u_reference["minimal_evidence_sets"] == []
if public_example:
    assert unanswerable["reference_package"]["qrels"] == {}
    print(
        "Expected response boundary:",
        unanswerable["reference_package"]["abstention_reason"],
    )

Expected response boundary: CHIMP defines SEND FOOD and server response types, but does not prescribe a numeric response timeout.


## 10. Check the important preparation boundaries

An empty page produces no chunk. Unsupported page status and OCR-required blockers
remain explicit, without silently changing the preparation method. These small
cases check the eligibility function with altered example records.

The RFC text also lets you verify that whitespace is collapsed while the source
content, headers, footers and split word at the page boundary are retained.

In [11]:
page = next(p for p in page_records if p["chunk_status"] == "chunk")
cases = [
    (page, "chunk"),
    (page | {"text": " \t\n "}, "skipped_empty_text"),
    (page | {"blocker": "blocked_ocr_required"}, "blocked_ocr_required"),
    (page | {"text_layer_status": "absent"}, "blocked_unsupported_status"),
    (page | {"extraction_status": "failed"}, "blocked_unsupported_status"),
]
for record, expected in cases:
    assert chunk_status(record) == expected
assert all("\n" not in c["text"] and "  " not in c["text"] for c in chunks)
if public_example:
    assert "[Page 1]" in chunks[0]["text"]
    assert "HEARTBEAT" in chunks[7]["text"]
    assert "KEEPER" in chunks[6]["text"] and "STOP" in chunks[6]["text"]
    assert "denial of service" in chunks[15]["text"]
    assert "may be copied" in chunks[19]["text"]
print("Eligibility, source text, page boundaries and reference links checked.")

Eligibility, source text, page boundaries and reference links checked.


## Prepared inputs for both system paths

The JSONL files preserve chunk/source order and accepted question wording.
Their input-only manifest records relative paths, hashes and source-document
identities. Deferred questions stay outside active generation. The accepted
reference packages remain in the source specification; its hash is retained.

Select `input_dir` as `data_dir` in Classical RAG or `prepared_dir` in GraphRAG.
Classical RAG uses `RUN_AZURE = True`; GraphRAG uses `RUN_AZURE_INDEX = True`. Choose a new run
folder for changed inputs. Saved public results can still be replayed with
these controls disabled. This preparation step makes no Azure request.


In [12]:
input_dir.mkdir(parents=True, exist_ok=True)
question_records = [
    {
        "question_id": q["question_id"],
        "canonical_question_id": q.get("canonical_question_id", q["question_id"]),
        "question_text": q["question_text"],
        "cohort": q["benchmark_role"],
    }
    for q in questions
    if q["benchmark_role"] != "deferred"
]
files = {
    "chunks": chunks,
    "chunk_origins": origins,
    "questions": question_records,
    "raw_pages": page_records,
}
artifacts = {}
for role, records in files.items():
    path = input_dir / f"{role}.jsonl"
    path.write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    artifacts[role] = {
        "path": path.name,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }
input_manifest = {
    "schema_version": "classical-rag-inputs-v1",
    "artifact_kind": "inputs",
    "chunking_policy": config.PAGE_CHUNKING_POLICY,
    "question_set_version": source["question_set_version"],
    "corpus_snapshot_id": snapshot_id,
    "source_documents": source_documents,
    "question_reference_sha256": hashlib.sha256(source_file.read_bytes()).hexdigest(),
    "artifacts": artifacts,
}
(input_dir / "run_manifest.json").write_text(
    json.dumps(input_manifest, indent=2) + "\n", encoding="utf-8"
)
print(
    {
        "prepared_input_folder": input_dir.as_posix(),
        "chunks": len(chunks),
        "questions": len(question_records),
    }
)

{'prepared_input_folder': '../local/classical_rag/rfc2795-inputs', 'chunks': 20, 'questions': 4}


## Prepared records and references

The prepared chunks and origin links serve both systems. Retain their source PDFs,
source specification and accepted references for source inspection and evaluation.
Private inputs, executed copies and generated artefacts remain in ignored `local/` folders.

Read [Classical RAG](02_classical_rag.ipynb) or
[GraphRAG indexing](03_graphrag_index.ipynb) to continue with the prepared inputs.
The [retrieval-metrics notebook](05_retrieval_metrics.ipynb) explains the separate
five-position ranking calculation and saved RFC results.

**Further reading:** thesis Sections 2.3–2.4 explain preparation, source evidence,
questions and reference packages; Section 2.9 explains the separate Genuine User
assessment. The [RFC Editor page](https://www.rfc-editor.org/info/rfc2795/) identifies
the public input document.
